### 1 - Load the chunks produced by 04-chunking

In [1]:
import json


def load_json(path):
    """Load a JSON file produced by an earlier notebook."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


chunks = load_json("chunks.json")
len(chunks)  # the chunk count from Q4

295

### 2 - Build the index over the chunks



In [2]:
from minsearch import Index


def build_index(records):
    """Create and fit a minsearch index (content=text, filename=keyword)."""
    index = Index(text_fields=["content"], keyword_fields=["filename"])
    index.fit(records)
    return index


chunk_index = build_index(chunks)

### 3 - Create the LLM client

In [3]:
from openai import OpenAI

llm_client = OpenAI()

### 4 - The RAG class (same self-contained class as Q3)


In [4]:
from dataclasses import dataclass

INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


@dataclass
class RAGResult:
    """Holds the final answer and the token usage of the request."""
    answer: str
    usage: object


class RAG:
    """Self-contained RAG over records with `filename` and `content`."""

    def __init__(self, index, llm_client, instructions=INSTRUCTIONS,
                 prompt_template=PROMPT_TEMPLATE, model="gpt-5.4-mini"):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(doc["filename"])
            lines.append(doc["content"])
            lines.append("")
        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(question=query, context=context)

    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt},
        ]
        return self.llm_client.responses.create(
            model=self.model, input=input_messages
        )

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return RAGResult(answer=response.output_text, usage=response.usage)

### 5 - Run the RAG over the chunk index (same query)

In [5]:
rag_chunked = RAG(index=chunk_index, llm_client=llm_client, model="gpt-5.4-mini")

query = "How does the agentic loop keep calling the model until it stops?"
result = rag_chunked.rag(query)

print(result.answer)

The loop keeps calling the model inside a `while True` loop. After each model response, it checks whether there were any `function_call` items:

- if there is at least one function call, it runs the tool, appends the result to `messages`, and loops again;
- if there are no function calls in that turn, it breaks out of the loop and stops.

So the stopping condition is:

```python
if has_function_calls == False:
    break
```

In other words, it keeps going until the model returns a final answer without asking for any more tools.


### 6 - Read the input tokens (same as Q3)

In [6]:
def get_input_tokens(usage):
    """Read input/prompt tokens regardless of SDK field name."""
    return getattr(usage, "input_tokens", None) or getattr(usage, "prompt_tokens", None)


tokens_q5 = get_input_tokens(result.usage)
tokens_q5

2294

### 7 - Answer to Q5: compare with Q3

In [15]:
tokens_q3 = 7111  

if tokens_q3:
    ratio = tokens_q3 / tokens_q5
    print(f"Q3 input tokens: {tokens_q3}")
    print(f"Q5 input tokens: {tokens_q5}")
    print(f"Q3 / Q5 ratio:   {ratio:.1f}x fewer")
else:
    print("Set tokens_q3 to your Q3 measurement to compute the ratio.")

Q3 input tokens: 7111
Q5 input tokens: 2294
Q3 / Q5 ratio:   3.1x fewer
